# 📊 Minhas Posições Atuais
Notebook que conecta ao Supabase e exibe as posições em carteira
com resumos acessíveis para leitores de tela.

## 1. Configuração e Conexão

In [1]:
import sys
print(f"Python: {sys.executable}")
print(f"Versão: {sys.version}")

Python: c:\Users\jcgerardi\Documents\Pessoais\financas\potfolio-tracker\.venv\Scripts\python.exe
Versão: 3.11.5 (tags/v3.11.5:cce6ba9, Aug 24 2023, 14:38:34) [MSC v.1936 64 bit (AMD64)]


In [2]:
import os
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client, Client

In [3]:
# Carrega .env
load_dotenv()

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

if not SUPABASE_URL or not SUPABASE_KEY:
    raise EnvironmentError("⚠️ SUPABASE_URL e SUPABASE_KEY precisam estar no .env")

db: Client = create_client(SUPABASE_URL, SUPABASE_KEY)
print("✅ Conectado ao Supabase com sucesso.")

✅ Conectado ao Supabase com sucesso.


## 2. Carregar Transações

In [4]:
response = (
    db.table("transactions")
    .select(
        "id, transaction_type, trade_date, quantity, unit_price, "
        "total_amount, total_amount_brl, exchange_rate_to_brl, "
        "currency_id, conversion_pair_id, "
        "assets!inner(id, ticker, name, asset_categories(name), currencies(code))"
    )
    .order("trade_date")
    .execute()
)

if not response.data:
    raise ValueError("⚠️ Nenhuma transação encontrada no Supabase.")

# Flatten das relações
records = []
for row in response.data:
    asset = row.pop("assets", {})
    category = asset.pop("asset_categories", {})
    currency = asset.pop("currencies", {})

    row["ticker"] = asset.get("ticker")
    row["asset_name"] = asset.get("name")
    row["categoria"] = category.get("name")
    row["moeda"] = currency.get("code")
    records.append(row)

df = pd.DataFrame(records)
df["trade_date"] = pd.to_datetime(df["trade_date"])

print(f"✅ {len(df)} transações carregadas.")
print(f"📅 Período: {df['trade_date'].min().strftime('%d/%m/%Y')} a {df['trade_date'].max().strftime('%d/%m/%Y')}")
print(f"🏷️ Ativos únicos: {df['ticker'].nunique()}")
print(f"📁 Categorias: {', '.join(df['categoria'].dropna().unique())}")

✅ 658 transações carregadas.
📅 Período: 08/03/2017 a 06/02/2026
🏷️ Ativos únicos: 113
📁 Categorias: Ações, Criptomoedas, FIIs, ETF Exterior, Tesouro, FIAGRO, Stocks, BDR


## 3. Calcular Posições (Saldo e Preço Médio)

In [5]:
def calcular_posicoes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula saldo acumulado e preço médio por ticker.
    
    Lógica:
    - Compra / Desdobramento / Bonificação / Conversão Entrada → soma quantidade
    - Venda / Conversão Saída → subtrai quantidade
    - Preço médio recalculado a cada compra (média ponderada)
    """
    
    TIPOS_ENTRADA = {"compra", "desdobramento", "bonificacao", "conversao_entrada"}
    TIPOS_SAIDA = {"venda", "conversao_saida"}

    posicoes = {}

    for _, row in df.sort_values("trade_date").iterrows():
        ticker = row["ticker"]
        tipo = row["transaction_type"]
        qtde = abs(row["quantity"])
        preco = abs(row["unit_price"]) if row["unit_price"] else 0
        moeda = row["moeda"]
        categoria = row["categoria"]
        nome = row["asset_name"]
        cambio = row.get("exchange_rate_to_brl") or 0
        total_brl = row.get("total_amount_brl") or 0

        if ticker not in posicoes:
            posicoes[ticker] = {
                "ticker": ticker,
                "nome": nome,
                "categoria": categoria,
                "moeda": moeda,
                "qtde_saldo": 0.0,
                "custo_total": 0.0,
                "preco_medio": 0.0,
                "custo_total_brl": 0.0,
                "preco_medio_brl": 0.0,
                "ultimo_cambio": cambio,
            }

        pos = posicoes[ticker]

        if tipo in TIPOS_ENTRADA:
            # Custo na moeda original
            custo_operacao = qtde * preco
            pos["custo_total"] += custo_operacao
            pos["qtde_saldo"] += qtde

            # Custo em BRL
            if moeda == "USD" and total_brl > 0:
                pos["custo_total_brl"] += abs(total_brl)
            elif moeda == "BRL":
                pos["custo_total_brl"] += custo_operacao

            if pos["qtde_saldo"] > 0:
                pos["preco_medio"] = pos["custo_total"] / pos["qtde_saldo"]
                pos["preco_medio_brl"] = pos["custo_total_brl"] / pos["qtde_saldo"]

            if cambio > 0:
                pos["ultimo_cambio"] = cambio

        elif tipo in TIPOS_SAIDA:
            if pos["qtde_saldo"] > 0:
                # Reduz custo proporcionalmente ao preço médio
                custo_reduzido = pos["preco_medio"] * qtde
                custo_reduzido_brl = pos["preco_medio_brl"] * qtde
                pos["custo_total"] -= custo_reduzido
                pos["custo_total_brl"] -= custo_reduzido_brl

            pos["qtde_saldo"] -= qtde

            # Se zerou, reseta
            if pos["qtde_saldo"] <= 0:
                pos["qtde_saldo"] = 0
                pos["custo_total"] = 0
                pos["custo_total_brl"] = 0
                pos["preco_medio"] = 0
                pos["preco_medio_brl"] = 0

    resultado = pd.DataFrame(posicoes.values())
    return resultado


posicoes = calcular_posicoes(df)
print(f"✅ Posições calculadas para {len(posicoes)} ativos.")

✅ Posições calculadas para 113 ativos.


## 4. Filtrar Posições Abertas (em carteira hoje)

In [8]:
abertas = posicoes[posicoes["qtde_saldo"] > 0].copy()
abertas.sort_values("categoria", inplace=True)
abertas.reset_index(drop=True, inplace=True)

print(f"✅ {len(abertas)} posições em aberto na carteira.\n")

✅ 75 posições em aberto na carteira.



## 5. Resumo Acessível — Posições por Categoria

In [9]:
def format_brl(value: float) -> str:
    """R$ 1.234,56"""
    if value >= 0:
        return f"R$ {value:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    return f"-R$ {abs(value):,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


def format_usd(value: float) -> str:
    """US$ 1,234.56"""
    if value >= 0:
        return f"US$ {value:,.2f}"
    return f"-US$ {abs(value):,.2f}"


def format_qtde(value: float) -> str:
    """Formata quantidade (inteiro se possível)."""
    if value == int(value):
        return f"{int(value)}"
    return f"{value:,.4f}".replace(",", "X").replace(".", ",").replace("X", ".")


# ——————————————————————————————————————————————
# RESUMO GERAL
# ——————————————————————————————————————————————

total_investido_brl = abertas["custo_total_brl"].sum()

print("=" * 60)
print("📊 RESUMO GERAL DA CARTEIRA")
print("=" * 60)
print(f"")
print(f"Total investido (BRL): {format_brl(total_investido_brl)}")
print(f"Posições abertas: {len(abertas)}")
print(f"Categorias: {abertas['categoria'].nunique()}")
print(f"")

# ——————————————————————————————————————————————
# POR CATEGORIA
# ——————————————————————————————————————————————

print("=" * 60)
print("📁 POSIÇÕES POR CATEGORIA")
print("=" * 60)

for categoria, grupo in abertas.groupby("categoria"):
    subtotal_brl = grupo["custo_total_brl"].sum()
    peso = (subtotal_brl / total_investido_brl * 100) if total_investido_brl > 0 else 0

    print(f"")
    print(f"── {categoria} ({len(grupo)} ativos | {peso:.1f}% da carteira | {format_brl(subtotal_brl)}) ──")
    print(f"")

    for _, row in grupo.sort_values("custo_total_brl", ascending=False).iterrows():
        peso_ativo = (row["custo_total_brl"] / total_investido_brl * 100) if total_investido_brl > 0 else 0

        if row["moeda"] == "USD":
            preco_info = (
                f"PM: {format_usd(row['preco_medio'])} "
                f"(BRL: {format_brl(row['preco_medio_brl'])}) | "
                f"Câmbio: R$ {row['ultimo_cambio']:.4f}"
            )
            custo_info = (
                f"Custo: {format_usd(row['custo_total'])} "
                f"({format_brl(row['custo_total_brl'])})"
            )
        else:
            preco_info = f"PM: {format_brl(row['preco_medio'])}"
            custo_info = f"Custo: {format_brl(row['custo_total_brl'])}"

        print(
            f"  • {row['ticker']:12s} | "
            f"Qtde: {format_qtde(row['qtde_saldo']):>10s} | "
            f"{preco_info} | "
            f"{custo_info} | "
            f"Peso: {peso_ativo:.1f}%"
        )

    print()

# ——————————————————————————————————————————————
# ALOCAÇÃO POR CATEGORIA
# ——————————————————————————————————————————————

print("=" * 60)
print("📊 ALOCAÇÃO POR CATEGORIA")
print("=" * 60)
print()

alocacao = (
    abertas.groupby("categoria")["custo_total_brl"]
    .sum()
    .sort_values(ascending=False)
)

for cat, valor in alocacao.items():
    peso = (valor / total_investido_brl * 100) if total_investido_brl > 0 else 0
    barra = "█" * int(peso / 2) + "░" * (50 - int(peso / 2))
    print(f"  {cat:20s} {barra} {peso:5.1f}% ({format_brl(valor)})")

print()
print(f"  {'TOTAL':20s} {'█' * 50} 100.0% ({format_brl(total_investido_brl)})")

# ——————————————————————————————————————————————
# ALOCAÇÃO POR MOEDA
# ——————————————————————————————————————————————

print()
print("=" * 60)
print("💱 ALOCAÇÃO POR MOEDA")
print("=" * 60)
print()

alocacao_moeda = (
    abertas.groupby("moeda")["custo_total_brl"]
    .sum()
    .sort_values(ascending=False)
)

for moeda, valor in alocacao_moeda.items():
    peso = (valor / total_investido_brl * 100) if total_investido_brl > 0 else 0
    barra = "█" * int(peso / 2) + "░" * (50 - int(peso / 2))
    print(f"  {moeda:20s} {barra} {peso:5.1f}% ({format_brl(valor)})")

📊 RESUMO GERAL DA CARTEIRA

Total investido (BRL): R$ 181.115,40
Posições abertas: 75
Categorias: 7

📁 POSIÇÕES POR CATEGORIA

── Ações (22 ativos | 49.8% da carteira | R$ 90.213,67) ──

  • ITSA4        | Qtde:       1138 | PM: R$ 11,36 | Custo: R$ 12.928,00 | Peso: 7.1%
  • USIM5        | Qtde:       1600 | PM: R$ 6,61 | Custo: R$ 10.580,47 | Peso: 5.8%
  • BBDC4        | Qtde:        520 | PM: R$ 13,49 | Custo: R$ 7.015,40 | Peso: 3.9%
  • CPLE3        | Qtde:        650 | PM: R$ 9,45 | Custo: R$ 6.144,50 | Peso: 3.4%
  • TTEN3        | Qtde:        500 | PM: R$ 12,21 | Custo: R$ 6.105,26 | Peso: 3.4%
  • SAPR11       | Qtde:        300 | PM: R$ 19,21 | Custo: R$ 5.763,30 | Peso: 3.2%
  • KEPL3        | Qtde:        700 | PM: R$ 7,96 | Custo: R$ 5.572,00 | Peso: 3.1%
  • BBAS3        | Qtde:        215 | PM: R$ 22,98 | Custo: R$ 4.940,45 | Peso: 2.7%
  • SHUL4        | Qtde:        800 | PM: R$ 5,41 | Custo: R$ 4.330,24 | Peso: 2.4%
  • GGBR4        | Qtde:        200 | PM: R$ 21,57

## 6. Tabela Completa — Todas as Posições Abertas

In [10]:
# DataFrame formatado para visualização
tabela = abertas[[
    "ticker", "nome", "categoria", "moeda",
    "qtde_saldo", "preco_medio", "preco_medio_brl",
    "custo_total", "custo_total_brl"
]].copy()

tabela.columns = [
    "Ticker", "Nome", "Categoria", "Moeda",
    "Quantidade", "PM (moeda)", "PM (BRL)",
    "Custo (moeda)", "Custo (BRL)"
]

tabela = tabela.sort_values("Custo (BRL)", ascending=False).reset_index(drop=True)

# Exibe
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 30)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

tabela

,Ticker,Nome,Categoria,Moeda,Quantidade,PM (moeda),PM (BRL),Custo (moeda),Custo (BRL)
0,IVV,IVV,ETF Exterior,USD,4.98,497.05,"2,611.43","2,476.65","13,011.96"
1,ITSA4,ITSA4,Ações,BRL,"1,138.00",11.36,11.36,"12,928.00","12,928.00"
2,USIM5,USIM5,Ações,BRL,"1,600.00",6.61,6.61,"10,580.47","10,580.47"
3,BBDC4,BBDC4,Ações,BRL,520.00,13.49,13.49,"7,015.40","7,015.40"
4,CPLE3,CPLE3,Ações,BRL,650.00,9.45,9.45,"6,144.50","6,144.50"
5,TTEN3,TTEN3,Ações,BRL,500.00,12.21,12.21,"6,105.26","6,105.26"
6,SAPR11,SAPR11,Ações,BRL,300.00,19.21,19.21,"5,763.30","5,763.30"
7,KEPL3,KEPL3,Ações,BRL,700.00,7.96,7.96,"5,572.00","5,572.00"
8,VISC11,VISC11,FIIs,BRL,42.00,130.75,130.75,"5,491.37","5,491.37"
9,TESOURO IPCA+ 2035,TESOURO IPCA+ 2035,Tesouro,BRL,2.20,"2,265.88","2,265.88","4,984.94","4,984.94"


## 7. Top 10 — Maiores Posições

In [11]:
top10 = abertas.nlargest(10, "custo_total_brl")

print("=" * 60)
print("🏆 TOP 10 MAIORES POSIÇÕES (por valor investido em BRL)")
print("=" * 60)
print()

for i, (_, row) in enumerate(top10.iterrows(), 1):
    peso = (row["custo_total_brl"] / total_investido_brl * 100) if total_investido_brl > 0 else 0
    print(
        f"  {i:2d}. {row['ticker']:12s} | "
        f"{row['categoria']:20s} | "
        f"{format_brl(row['custo_total_brl']):>16s} | "
        f"Peso: {peso:.1f}%"
    )

🏆 TOP 10 MAIORES POSIÇÕES (por valor investido em BRL)

   1. IVV          | ETF Exterior         |     R$ 13.011,96 | Peso: 7.2%
   2. ITSA4        | Ações                |     R$ 12.928,00 | Peso: 7.1%
   3. USIM5        | Ações                |     R$ 10.580,47 | Peso: 5.8%
   4. BBDC4        | Ações                |      R$ 7.015,40 | Peso: 3.9%
   5. CPLE3        | Ações                |      R$ 6.144,50 | Peso: 3.4%
   6. TTEN3        | Ações                |      R$ 6.105,26 | Peso: 3.4%
   7. SAPR11       | Ações                |      R$ 5.763,30 | Peso: 3.2%
   8. KEPL3        | Ações                |      R$ 5.572,00 | Peso: 3.1%
   9. VISC11       | FIIs                 |      R$ 5.491,37 | Peso: 3.0%
  10. TESOURO IPCA+ 2035 | Tesouro              |      R$ 4.984,94 | Peso: 2.8%


## 8. Posições Encerradas (saldo = 0)

In [12]:
encerradas = posicoes[posicoes["qtde_saldo"] == 0].copy()

if len(encerradas) > 0:
    print(f"📋 {len(encerradas)} posições encerradas (já vendidas/convertidas):\n")
    for _, row in encerradas.iterrows():
        print(f"  ✖ {row['ticker']:12s} | {row['categoria']}")
else:
    print("📋 Nenhuma posição encerrada encontrada.")

📋 38 posições encerradas (já vendidas/convertidas):

  ✖ CSUD3        | Ações
  ✖ TECN3        | Ações
  ✖ PTBL3        | Ações
  ✖ OGXP3        | Ações
  ✖ FRAS3        | Ações
  ✖ TEND3        | Ações
  ✖ PETR4        | Ações
  ✖ XPLG11       | FIIs
  ✖ JSRE11       | FIIs
  ✖ RBRF11       | FIIs
  ✖ KLBN3        | Ações
  ✖ CPLE6        | Ações
  ✖ TAEE4        | Ações
  ✖ DEVA11       | FIIs
  ✖ GOVT         | ETF Exterior
  ✖ HCTR11       | FIIs
  ✖ AESB3        | Ações
  ✖ SIMH3        | Ações
  ✖ IRBR3        | Ações
  ✖ KLBN11       | Ações
  ✖ EUCA4        | Ações
  ✖ TAEE11       | Ações
  ✖ EZTC3        | Ações
  ✖ HSLG11       | FIIs
  ✖ MRVE3        | Ações
  ✖ MILS3        | Ações
  ✖ BOTZ         | ETF Exterior
  ✖ RNEW11       | Ações
  ✖ BRAV3        | Ações
  ✖ WEGE3        | Ações
  ✖ MGLU3        | Ações
  ✖ RAIZ4        | Ações
  ✖ AAVE         | Criptomoedas
  ✖ AGIX         | Criptomoedas
  ✖ FESA4        | Ações
  ✖ CNES11       | FIIs
  ✖ ASML34       | BDR
  ✖